# Анализ динамики городской застройки — г. Ташкент
**Спутник:** Sentinel-2 L2A  
**Период:** 2019 — 2026  
**Индексы:** NDBI, NDVI, MNDWI, BUI  
**Метод:** STAC API + COG windowed reading  

Пайплайн:
1. Загрузка каналов (выполнена `download_sentinel2.py`)
2. Расчёт индексов
3. Маскирование воды и растительности
4. Оценка площади застройки (га)
5. Временной ряд
6. Визуальная валидация (True Color + маска)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import rasterio
from rasterio.enums import Resampling
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from tqdm.notebook import tqdm

## Конфигурация
Все настройки в одном месте — менять только здесь.

In [ ]:
config = {
    # папка со скачанными .tif файлами
    "data_dir": Path("data/sentinel2"),

    # рабочее разрешение пикселя (метры)
    # B11/B12 — 20м, B02/B03/B04/B08 — 10м → приводим всё к 20м
    "resolution": 20,

    # пороговые значения для масок
    "ndbi_threshold":  0.0,   # NDBI > 0 → застройка
    "ndvi_threshold":  0.2,   # NDVI > 0.2 → растительность (исключаем)
    "mndwi_threshold": 0.0,   # MNDWI > 0 → вода (исключаем)

    # имена каналов (соответствуют именам файлов)
    "bands": {
        "blue":  "B02",
        "green": "B03",
        "red":   "B04",
        "nir":   "B08",
        "swir1": "B11",
        "swir2": "B12",
    },
}

## Вспомогательные функции

In [ ]:
def get_available_dates(data_dir: Path, required_bands=("B11", "B08", "B04", "B03")) -> list[str]:
    """
    Находит даты, для которых скачаны все нужные каналы.
    required_bands — минимальный набор для расчёта NDBI + NDVI + MNDWI.
    """
    from collections import defaultdict
    dates_bands = defaultdict(set)
    for f in data_dir.glob("s2_B*.tif"):
        # имя файла: s2_B11_2022-07-14.tif
        parts = f.stem.split("_")  # ['s2', 'B11', '2022-07-14']
        if len(parts) == 3:
            band, date = parts[1], parts[2]
            dates_bands[date].add(band)

    complete = [
        d for d, bands in dates_bands.items()
        if all(b in bands for b in required_bands)
    ]
    return sorted(complete)


def load_band(date: str, band: str, data_dir: Path, target_shape=None) -> np.ndarray:
    """
    Загружает канал за дату, опционально ресемплирует до target_shape.
    Возвращает float32 массив с NaN вместо nodata.
    """
    path = data_dir / f"s2_{band}_{date}.tif"
    with rasterio.open(path) as src:
        if target_shape is not None and src.shape != target_shape:
            data = src.read(
                1,
                out_shape=(1, *target_shape),
                resampling=Resampling.bilinear,
            ).astype(np.float32)
        else:
            data = src.read(1).astype(np.float32)

        # заменяем nodata и нули на NaN
        nodata = src.nodata
        if nodata is not None:
            data[data == nodata] = np.nan
        data[data == 0] = np.nan

    return data


def get_reference_shape(date: str, data_dir: Path, band="B11") -> tuple:
    """Возвращает (height, width) опорного канала для ресемплинга."""
    path = data_dir / f"s2_{band}_{date}.tif"
    with rasterio.open(path) as src:
        return src.shape  # (height, width)


def safe_index(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """(a - b) / (a + b) с защитой от деления на ноль."""
    with np.errstate(invalid="ignore", divide="ignore"):
        result = (a - b) / (a + b)
    result[~np.isfinite(result)] = np.nan
    return result


def calc_indices(date: str, data_dir: Path, cfg: dict) -> dict | None:
    """
    Рассчитывает NDBI, NDVI, MNDWI, BUI для одной даты.
    Возвращает словарь с массивами индексов или None если файлов нет.
    """
    try:
        shape = get_reference_shape(date, data_dir)
        b = cfg["bands"]

        B11 = load_band(date, b["swir1"], data_dir, shape)
        B08 = load_band(date, b["nir"],   data_dir, shape)
        B04 = load_band(date, b["red"],   data_dir, shape)
        B03 = load_band(date, b["green"], data_dir, shape)

        ndbi  = safe_index(B11, B08)  # (B11 - B08) / (B11 + B08)
        ndvi  = safe_index(B08, B04)  # (B08 - B04) / (B08 + B04)
        mndwi = safe_index(B03, B11)  # (B03 - B11) / (B03 + B11)
        bui   = ndbi - ndvi

        return {"ndbi": ndbi, "ndvi": ndvi, "mndwi": mndwi, "bui": bui, "shape": shape}

    except FileNotFoundError as e:
        print(f"  [!] {date}: не найден файл — {e.filename}")
        return None


def calc_builtup_area_ha(ndbi, ndvi, mndwi, cfg) -> float:
    """
    Считает площадь застройки в гектарах.
    Маскируем воду и растительность — остаётся чистая застройка.
    """
    res = cfg["resolution"]
    pixel_area_ha = (res * res) / 10_000  # м² → га

    builtup_mask = (
        (ndbi  >  cfg["ndbi_threshold"])  &
        (ndvi  <  cfg["ndvi_threshold"])  &   # исключаем растительность
        (mndwi <  cfg["mndwi_threshold"]) &   # исключаем воду
        np.isfinite(ndbi)
    )
    return float(builtup_mask.sum() * pixel_area_ha)

## Доступные даты
Смотрим сколько сцен уже скачано и готово к анализу.

In [ ]:
dates = get_available_dates(config["data_dir"])
print(f"Готово к анализу: {len(dates)} сцен")
print(f"Первая: {dates[0]}  |  Последняя: {dates[-1]}")

## Расчёт индексов и площади застройки
Проходим по всем датам, считаем NDBI, применяем маски воды и растительности, 
переводим пиксели в гектары.

In [ ]:
results = []  # список словарей {date, area_ha}

for date in tqdm(dates, desc="Обработка сцен"):
    idx = calc_indices(date, config["data_dir"], config)
    if idx is None:
        continue

    area = calc_builtup_area_ha(
        idx["ndbi"], idx["ndvi"], idx["mndwi"], config
    )
    results.append({
        "date": datetime.strptime(date, "%Y-%m-%d"),
        "date_str": date,
        "area_ha": area,
    })

print(f"\nОбработано сцен: {len(results)}")
if results:
    areas = [r["area_ha"] for r in results]
    print(f"Мин. площадь : {min(areas):.1f} га")
    print(f"Макс. площадь: {max(areas):.1f} га")

## Временной ряд площади застройки
График показывает как менялась площадь застройки в Ташкенте с 2019 года.

In [ ]:
if not results:
    print("Нет данных для графика. Дождитесь окончания загрузки.")
else:
    dates_dt = [r["date"] for r in results]
    areas    = [r["area_ha"] for r in results]

    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(dates_dt, areas, "o-", color="#e74c3c", linewidth=1.5,
            markersize=5, label="Площадь застройки (NDBI)")

    # линия тренда
    x_num = mdates.date2num(dates_dt)
    z = np.polyfit(x_num, areas, 1)
    p = np.poly1d(z)
    ax.plot(dates_dt, p(x_num), "--", color="#c0392b", alpha=0.6, label="Тренд")

    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_xlabel("Год")
    ax.set_ylabel("Площадь, га")
    ax.set_title("Динамика площади застройки — г. Ташкент (Sentinel-2, NDBI)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("timeseries_builtup.png", dpi=150)
    plt.show()
    print("График сохранён: timeseries_builtup.png")

## Визуальная валидация
Преподаватель рекомендовал **обязательно** показывать True Color рядом с маской —
чтобы убедиться что NDBI не захватывает артефакты (голая почва, облака).

Выбираем 4 даты: по одной из каждого сезона.

In [ ]:
def load_rgb(date: str, data_dir: Path, cfg: dict, shape=None) -> np.ndarray:
    """Загружает True Color (B04/B03/B02), нормирует в [0, 1]."""
    b = cfg["bands"]
    R = load_band(date, b["red"],   data_dir, shape)
    G = load_band(date, b["green"], data_dir, shape)
    B = load_band(date, b["blue"],  data_dir, shape)

    rgb = np.dstack([R, G, B]).astype(float)
    p2, p98 = np.nanpercentile(rgb, 2), np.nanpercentile(rgb, 98)
    rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-10), 0, 1)
    rgb = np.nan_to_num(rgb, nan=0.0)
    return rgb


def plot_validation(date: str, data_dir: Path, cfg: dict):
    """True Color + NDBI-маска застройки рядом для одной даты."""
    idx = calc_indices(date, data_dir, cfg)
    if idx is None:
        return

    shape = idx["shape"]
    rgb = load_rgb(date, data_dir, cfg, shape)

    builtup = (
        (idx["ndbi"]  > cfg["ndbi_threshold"])  &
        (idx["ndvi"]  < cfg["ndvi_threshold"])  &
        (idx["mndwi"] < cfg["mndwi_threshold"]) &
        np.isfinite(idx["ndbi"])
    )
    area = builtup.sum() * (cfg["resolution"] ** 2) / 10_000

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # True Color
    axes[0].imshow(rgb)
    axes[0].set_title(f"True Color\n{date}")
    axes[0].axis("off")

    # NDBI
    im = axes[1].imshow(idx["ndbi"], cmap="RdYlGn_r", vmin=-0.5, vmax=0.5)
    axes[1].set_title("NDBI\n(красный = застройка)")
    axes[1].axis("off")
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    # Маска застройки
    overlay = rgb.copy()
    overlay[builtup, 0] = 1.0   # красный канал
    overlay[builtup, 1] = 0.2
    overlay[builtup, 2] = 0.2
    axes[2].imshow(overlay)
    axes[2].set_title(f"Маска застройки\n{area:.1f} га")
    axes[2].axis("off")

    plt.suptitle(f"Ташкент — {date}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"validation_{date}.png", dpi=150)
    plt.show()

In [ ]:
# выбираем даты для визуальной проверки
# если данных мало — берём что есть
if len(dates) == 0:
    print("Данных пока нет — дождись окончания загрузки")
elif len(dates) <= 4:
    check_dates = dates
else:
    # берём примерно равномерно по временному ряду
    step = len(dates) // 4
    check_dates = [dates[0], dates[step], dates[step*2], dates[-1]]

for d in check_dates:
    plot_validation(d, config["data_dir"], config)

## Сезонное сравнение
Сравниваем лето vs зима — NDBI ведёт себя по-разному в зависимости от сезона
(снег зимой может имитировать застройку).

In [ ]:
# группируем по сезонам
summer = [r for r in results if r["date"].month in (6, 7, 8)]
winter = [r for r in results if r["date"].month in (12, 1, 2)]

if summer and winter:
    avg_summer = np.mean([r["area_ha"] for r in summer])
    avg_winter = np.mean([r["area_ha"] for r in winter])

    print(f"Среднее лето  (июнь–август):  {avg_summer:.1f} га")
    print(f"Среднее зима  (дек–февраль):  {avg_winter:.1f} га")
    print(f"Разница (артефакты зимой?):   {abs(avg_winter - avg_summer):.1f} га")

    fig, ax = plt.subplots(figsize=(14, 5))
    if summer:
        ax.scatter([r["date"] for r in summer], [r["area_ha"] for r in summer],
                   color="#f39c12", label="Лето", zorder=5, s=60)
    if winter:
        ax.scatter([r["date"] for r in winter], [r["area_ha"] for r in winter],
                   color="#3498db", label="Зима", zorder=5, s=60)

    ax.plot([r["date"] for r in results], [r["area_ha"] for r in results],
            "-", color="gray", alpha=0.4, linewidth=1)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("Площадь, га")
    ax.set_title("Сезонное сравнение — Ташкент")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("seasonal_comparison.png", dpi=150)
    plt.show()
else:
    print("Недостаточно данных для сезонного сравнения")

## Итоговая таблица
Площадь застройки по каждой дате.

In [ ]:
print(f"{'Дата':<14} {'Площадь, га':>12}")
print("-" * 28)
for r in results:
    print(f"{r['date_str']:<14} {r['area_ha']:>12.1f}")